In [1]:
# !pip install langchain langchain-community langchain-google-genai PyPDF2 langchain-classic pandas 
# !pip install langchain-huggingface


In [2]:
import os
import json
import pandas as pd
import traceback
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [4]:
# GET THE API KEY FROM HERE : https://aistudio.google.com/
KEY = "YOUR_GOOGLE_AI_STUDIO_API_KEY_HERE"

In [5]:
os.environ["GOOGLE_API_KEY"] = KEY

In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0.5,
    google_api_key=KEY
)

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_classic.chains import SequentialChain
import PyPDF2

In [8]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [9]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [10]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
    )

In [11]:
quiz_chain=LLMChain(llm=llm, prompt=quiz_generation_prompt, output_key="quiz", verbose=True)

C:\Users\shrey\AppData\Local\Temp\ipykernel_11632\2669661367.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  quiz_chain=LLMChain(llm=llm, prompt=quiz_generation_prompt, output_key="quiz", verbose=True)


In [12]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [13]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE)

In [14]:
review_chain=LLMChain(llm=llm, prompt=quiz_evaluation_prompt, output_key="review", verbose=True)

In [15]:
generate_evaluate_chain=SequentialChain(chains=[quiz_chain, review_chain], input_variables=["text", "number", "subject", "tone", "response_json"],
                                        output_variables=["quiz", "review"], verbose=True,)

In [16]:
file_path=r"D:\Developement\GEN AI\Gen AI - Projects\01_data.txt"

In [17]:
file_path

'D:\\Developement\\GEN AI\\Gen AI - Projects\\01_data.txt'

In [18]:
with open(file_path, 'r', encoding='utf-8') as file:
    TEXT = file.read()

In [19]:
print(TEXT)

What is machine learning?
Machine learning is the subset of artificial intelligence (AI) focused on algorithms that can “learn” the patterns of training data and, subsequently, make accurate inferences about new data. This pattern recognition ability enables machine learning models to make decisions or predictions without explicit, hard-coded instructions.

Machine learning has come to dominate the field of AI: it provides the backbone of most modern AI systems, from forecasting models to autonomous vehicles to large language models (LLMs) and other generative AI tools.

The central premise of machine learning (ML) is that if you optimize a model’s performance on a dataset of tasks that adequately resemble the real-world problems it will be used for—through a process called model training—the model can make accurate predictions on the new data it sees in its ultimate use case.

Training itself is simply a means to an end: generalization, the translation of strong performance on trainin

In [20]:
# Serialize the Python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [21]:
NUMBER=5 
SUBJECT="biology"
TONE="simple"

In [22]:
response = generate_evaluate_chain.invoke(
    {
        "text": TEXT,
        "number": NUMBER,
        "subject": SUBJECT,
        "tone": TONE,
        "response_json": json.dumps(RESPONSE_JSON)
    }
)



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Text:What is machine learning?
Machine learning is the subset of artificial intelligence (AI) focused on algorithms that can “learn” the patterns of training data and, subsequently, make accurate inferences about new data. This pattern recognition ability enables machine learning models to make decisions or predictions without explicit, hard-coded instructions.

Machine learning has come to dominate the field of AI: it provides the backbone of most modern AI systems, from forecasting models to autonomous vehicles to large language models (LLMs) and other generative AI tools.

The central premise of machine learning (ML) is that if you optimize a model’s performance on a dataset of tasks that adequately resemble the real-world problems it will be used for—through a process called model training—the model can make accurate predictions on the new data it sees in its ultimate use case.

T

In [23]:
print(response)

{'text': 'What is machine learning?\nMachine learning is the subset of artificial intelligence (AI) focused on algorithms that can “learn” the patterns of training data and, subsequently, make accurate inferences about new data. This pattern recognition ability enables machine learning models to make decisions or predictions without explicit, hard-coded instructions.\n\nMachine learning has come to dominate the field of AI: it provides the backbone of most modern AI systems, from forecasting models to autonomous vehicles to large language models (LLMs) and other generative AI tools.\n\nThe central premise of machine learning (ML) is that if you optimize a model’s performance on a dataset of tasks that adequately resemble the real-world problems it will be used for—through a process called model training—the model can make accurate predictions on the new data it sees in its ultimate use case.\n\nTraining itself is simply a means to an end: generalization, the translation of strong perfo

In [24]:
quiz=response.get("quiz")

In [25]:
# 1. Get the quiz string from your response dictionary
quiz_output = response.get("quiz")

# 2. Clean the Markdown backticks and the 'json' tag
# This removes the starting ```json and the ending ```
cleaned_quiz = quiz_output.replace("```json", "").replace("```", "").strip()

# 3. Now you can safely use json.loads
try:
    quiz_json = json.loads(cleaned_quiz)
    print("✅ Successfully parsed JSON!")
    # Proceed with your logic using quiz_json
except json.JSONDecodeError as e:
    print(f"❌ Failed to parse: {e}")

✅ Successfully parsed JSON!


In [26]:
import json

# 1. Clean and Convert the string to a Dictionary
# We extract 'quiz' from the response and remove markdown backticks
quiz_raw_str = response.get("quiz")
quiz_cleaned = quiz_raw_str.replace("```json", "").replace("```", "").strip()

# Convert to dictionary
quiz_dict = json.loads(quiz_cleaned)

# 2. Now run your table extraction logic on the DICTIONARY (quiz_dict)
quiz_table_data = []
for key, value in quiz_dict.items(): # Use quiz_dict here!
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
        ]
    )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

# 3. Create and display the DataFrame
import pandas as pd
df = pd.DataFrame(quiz_table_data)
display(df)

,MCQ,Choices,Correct
0,What is the main idea behind machine learning?,a: Giving computers a very long list of exact ...,b
1,How does machine learning relate to Artificial...,a: Machine learning is completely separate fro...,b
2,What is the main goal when training a machine ...,a: To make the model perfectly remember all th...,b
3,"Which statement best describes deep learning, ...","a: It's an older, simpler type of machine lear...",b
4,Who is often given credit for first using the ...,a: The team of scientists who developed the fi...,b


In [27]:
quiz_table_data

[{'MCQ': 'What is the main idea behind machine learning?',
  'Choices': 'a: Giving computers a very long list of exact instructions for every task. | b: Allowing computers to learn patterns from data so they can make smart guesses or predictions. | c: Making computers only good at playing games, like checkers. | d: Helping humans write more complicated computer code. | e: Machine learning is a type of artificial intelligence (AI) that teaches computers to learn from data, make predictions, and make decisions without being explicitly programmed.',
  'Correct': 'b'},
 {'MCQ': 'How does machine learning relate to Artificial Intelligence (AI)?',
  'Choices': "a: Machine learning is completely separate from AI. | b: Machine learning is a smaller part or 'subset' of AI. | c: AI is a smaller part or 'subset' of machine learning. | d: They are just two different names for the exact same thing.",
  'Correct': 'b'},
 {'MCQ': 'What is the main goal when training a machine learning model?',
  'Cho

In [28]:
quiz=pd.DataFrame(quiz_table_data)

In [29]:
quiz.to_csv("01_machinelearning.csv",index=False)